# Lab 2: Data Preparation and Feature Engineering

**DSA 8401 Applied Machine Learning | Chapter 2**

Dataset: `mobile_money_statements.csv` | 183,600 transactions | 18 columns

## Table of Contents

1. Environment Setup
2. Raw Data Profile
3. Data Quality Audit (10 documented issues)
4. Data Cleaning and Parsing
5. Feature Engineering (28 features, 5 families)
6. Leakage Assessment
7. Reproducible ML Pipeline
8. Final Fit and Model Persistence
9. Reproducibility Record
10. Technical Report (Head of Credit Risk)

In [1]:
import re
import warnings
import sys
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

SCORING_TS   = pd.Timestamp('2026-08-01 00:00:00')
DATA_PATH    = 'mobile_money_statements.csv'
MODEL_PATH   = 'final_pipeline.joblib'
RANDOM_STATE = 42

print(f'Scoring timestamp : {SCORING_TS}')
print(f'Dataset           : {DATA_PATH}')
print(f'Random state seed : {RANDOM_STATE}')

Scoring timestamp : 2026-08-01 00:00:00
Dataset           : mobile_money_statements.csv
Random state seed : 42


In [2]:
raw = pd.read_csv(DATA_PATH)
print(f'Shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns')
print('\nColumn dtypes:')
print(raw.dtypes.to_string())
raw.head(3).T

Shape: 183,600 rows x 18 columns

Column dtypes:
txn_id                  object
msisdn                   int64
reg_id                  object
account_name            object
region                  object
segment                 object
txn_time                object
txn_type                object
amount                  object
gps_lat                float64
gps_lon                float64
agent_id                object
device_id               object
counterparty            object
balance_after          float64
manual_review_score    float64
settlement_status       object
is_fraud                 int64


,0,1,2
txn_id,TXN0102856,TXN0155490,TXN0019434
msisdn,254700004022,254700013612,254700005862
reg_id,NID101089,NID103658,nid101584
account_name,Joseph Mwangi,Rehema Ndlovu,Mary Mutua
region,Kampala,Mombasa,Arusha
segment,retail,retail,retail
txn_time,2026-06-30 14:24:19,"Jan 29, 2026 01:17 PM","Sep 16, 2025 12:55 AM"
txn_type,send,send,receive
amount,"1,329/- Dr",289/- Dr,"1,571/-"
gps_lat,NaN,1.7747,-3.7434


## 1. Data Quality Audit

Each issue is documented with evidence, missingness classification (where applicable), remediation, and rationale.

**Missingness mechanism definitions:**
- **MCAR** (Missing Completely At Random): missingness is unrelated to any observed or unobserved variable
- **MAR** (Missing At Random): missingness is fully explainable by other observed columns
- **MNAR** (Missing Not At Random): missingness depends on the unobserved value itself

In [3]:
miss = raw.isna().mean().sort_values(ascending=False)
miss_df = pd.DataFrame({
    'missing_frac': miss[miss > 0].round(4),
    'missing_n': (miss[miss > 0] * len(raw)).astype(int)
})
print('Columns with missing values:')
print(miss_df.to_string())
print(f'\nColumns with zero missingness: {(miss == 0).sum()} of {len(miss)}')

Columns with missing values:
           missing_frac  missing_n
gps_lon          0.3621      66483
gps_lat          0.3621      66483
agent_id         0.1369      25142
device_id        0.0591      10856

Columns with zero missingness: 14 of 18


### Issue 1: Exact Duplicate Rows

In [4]:
n_exact   = int(raw.duplicated().sum())
n_txn_dup = int(raw['txn_id'].duplicated().sum())
n_rows    = len(raw)

print(f'Total rows           : {n_rows:,}')
print(f'Exact duplicate rows : {n_exact:,}  ({n_exact/n_rows*100:.2f}%)')
print(f'Duplicate txn_id     : {n_txn_dup:,}')

# Confirm all duplicate txn_ids are byte-for-byte row copies
dup_txn_mask = raw['txn_id'].duplicated(keep=False)
pct_exact_match = raw[dup_txn_mask].duplicated(keep=False).mean() * 100
print(f'Byte-identical within dup txn_ids: {pct_exact_match:.1f}%')

Total rows           : 183,600
Exact duplicate rows : 3,600  (1.96%)
Duplicate txn_id     : 3,600
Byte-identical within dup txn_ids: 100.0%


| Component | Detail |
|---|---|
| **Issue** | 3,600 exact duplicate rows (2.0% of dataset) |
| **Evidence** | `raw.duplicated().sum() == 3,600`; duplicate `txn_id` values are 100% byte-identical |
| **Mechanism** | Batch-export retry replayed the last window of events |
| **Remediation** | `drop_duplicates()` before any split or aggregation |
| **Rationale** | Retaining duplicates would leak the same event into both train and validation folds |

### Issue 2: `amount` — Three Competing Debit Encodings

In [5]:
s        = raw['amount'].astype(str)
is_dr    = s.str.contains('Dr', na=False)
is_paren = s.str.startswith('(')
is_neg   = s.str.startswith('-')
is_plain = ~is_dr & ~is_paren & ~is_neg

print('Amount format distribution:')
print(f'  Plain credit  (e.g. 1,500/-)    : {is_plain.sum():>7,}  ({is_plain.mean()*100:5.1f}%)')
print(f'  Debit Dr      (e.g. 1,500/- Dr) : {is_dr.sum():>7,}  ({is_dr.mean()*100:5.1f}%)')
print(f'  Debit paren   (e.g. (1,500/-))  : {is_paren.sum():>7,}  ({is_paren.mean()*100:5.1f}%)')
print(f'  Debit minus   (e.g. -1,500/-)   : {is_neg.sum():>7,}  ({is_neg.mean()*100:5.1f}%)')

print('\nOne sample of each format:')
for label, mask in [('Credit', is_plain), ('Dr', is_dr), ('Paren', is_paren), ('Minus', is_neg)]:
    val = s[mask].iloc[0] if mask.any() else 'N/A'
    print(f'  {label:<8}: {val}')

Amount format distribution:
  Plain credit  (e.g. 1,500/-)    :  62,104  ( 33.8%)
  Debit Dr      (e.g. 1,500/- Dr) :  40,391  ( 22.0%)
  Debit paren   (e.g. (1,500/-))  :  40,635  ( 22.1%)
  Debit minus   (e.g. -1,500/-)   :  40,470  ( 22.0%)

One sample of each format:
  Credit  : 1,571/-
  Dr      : 1,329/- Dr
  Paren   : (1,307/-)
  Minus   : -2,136/-


| Component | Detail |
|---|---|
| **Issue** | `amount` is a string with three distinct debit encodings and a currency suffix (`/-`) |
| **Evidence** | `dtype == object`; samples show Dr-suffix, parentheses, and minus-prefix variants |
| **Mechanism** | Three regional billing systems exported with different debit sign conventions |
| **Remediation** | `parse_amount()` function that strips formatting and applies a unified sign rule |
| **Rationale** | A single deterministic rule correctly recovers the signed float for all observed formats |

### Issue 3: `txn_time` — Mixed Datetime Formats

In [6]:
s = raw['txn_time'].astype(str)

fmt_iso = pd.to_datetime(s, format='%Y-%m-%d %H:%M:%S', errors='coerce').notna()
fmt_eur = pd.to_datetime(s, format='%d/%m/%Y %H:%M',    errors='coerce').notna()
fmt_us  = pd.to_datetime(s, format='%b %d, %Y %I:%M %p', errors='coerce').notna()
unmatched = ~fmt_iso & ~fmt_eur & ~fmt_us

print('txn_time format distribution:')
print(f'  ISO    (YYYY-MM-DD HH:MM:SS)  : {fmt_iso.sum():>7,}  ({fmt_iso.mean()*100:5.1f}%)')
print(f'  Europe (DD/MM/YYYY HH:MM)     : {fmt_eur.sum():>7,}  ({fmt_eur.mean()*100:5.1f}%)')
print(f'  US     (Mon DD, YYYY HH:MM)   : {fmt_us.sum():>7,}  ({fmt_us.mean()*100:5.1f}%)')
print(f'  Unparsed by all three         : {unmatched.sum():>7,}')

print('\nSamples:')
for label, mask in [('ISO', fmt_iso), ('Europe', fmt_eur), ('US', fmt_us)]:
    print(f'  {label:<8}: {s[mask].iloc[0]}')

txn_time format distribution:
  ISO    (YYYY-MM-DD HH:MM:SS)  :  61,167  ( 33.3%)
  Europe (DD/MM/YYYY HH:MM)     :  61,173  ( 33.3%)
  US     (Mon DD, YYYY HH:MM)   :  61,260  ( 33.4%)
  Unparsed by all three         :       0

Samples:
  ISO     : 2026-06-30 14:24:19
  Europe  : 07/04/2026 11:21
  US      : Jan 29, 2026 01:17 PM


| Component | Detail |
|---|---|
| **Issue** | `txn_time` mixes three datetime formats in a single column |
| **Evidence** | Single-pass `pd.to_datetime` fails on ~67% of rows; three-pass parsing succeeds on all |
| **Mechanism** | Three regional sub-systems export timestamps in different locale conventions |
| **Remediation** | Sequential multi-pass parsing: ISO then European then US-locale |
| **Rationale** | Multi-pass with `errors='coerce'` guarantees all rows are parsed without data loss |

### Issue 4: `reg_id` — Case and Whitespace Inconsistency

In [7]:
n_raw   = raw['reg_id'].nunique()
n_clean = raw['reg_id'].str.strip().str.upper().nunique()
n_sim   = raw['msisdn'].nunique()

print(f'Unique SIM numbers (msisdn)       : {n_sim:>6,}')
print(f'Unique reg_id (raw)               : {n_raw:>6,}')
print(f'Unique reg_id (strip + upper)     : {n_clean:>6,}')
print(f'Apparent duplicates removed       : {n_raw - n_clean:>6,}')

dirty_mask = raw['reg_id'] != raw['reg_id'].str.strip().str.upper()
dirty_vals = raw.loc[dirty_mask, 'reg_id'].unique()
print('\nDirty reg_id examples (first 6):')
for d in dirty_vals[:6]:
    print(f'  {repr(d):<22}  ->  {repr(d.strip().upper())}')

Unique SIM numbers (msisdn)       :  4,825
Unique reg_id (raw)               : 11,722
Unique reg_id (strip + upper)     :  3,991
Apparent duplicates removed       :  7,731



Dirty reg_id examples (first 6):
  'nid101584'             ->  'NID101584'
  'nid103963'             ->  'NID103963'
  '  NID101375 '          ->  'NID101375'
  'nid103172'             ->  'NID103172'
  '  NID103820 '          ->  'NID103820'
  '  NID102258 '          ->  'NID102258'


| Component | Detail |
|---|---|
| **Issue** | `reg_id` appears in mixed case and with leading/trailing whitespace |
| **Evidence** | 11,722 raw unique IDs collapse to 3,991 after `.strip().upper()` |
| **Mechanism** | Multiple data entry paths with no upstream normalisation |
| **Remediation** | `df['customer_id'] = df['reg_id'].str.strip().str.upper()` |
| **Rationale** | `customer_id` is the `GroupKFold` key; failure to resolve scatters one person's SIMs across folds |

### Issue 5: `account_name` — HTML Entities and Mixed Casing

In [8]:
has_nbsp  = raw['account_name'].str.contains('\xa0', na=False)
all_caps  = raw['account_name'].str.isupper()
comma_fmt = raw['account_name'].str.contains(', ', na=False)

print('account_name quality issues:')
print(f'  HTML non-breaking space (\\xa0) : {has_nbsp.sum():>7,}  ({has_nbsp.mean()*100:5.1f}%)')
print(f'  All-caps names                  : {all_caps.sum():>7,}  ({all_caps.mean()*100:5.1f}%)')
print(f'  Surname-comma-first format      : {comma_fmt.sum():>7,}  ({comma_fmt.mean()*100:5.1f}%)')

dirty = raw.loc[has_nbsp | all_caps | comma_fmt, 'account_name'].unique()
print('\nSamples of dirty names:')
for n in dirty[:8]:
    print(f'  {repr(n)}')

account_name quality issues:
  HTML non-breaking space (\xa0) :       0  (  0.0%)
  All-caps names                  :  30,523  ( 16.6%)
  Surname-comma-first format      :  30,680  ( 16.7%)

Samples of dirty names:
  'SSENTONGO GRACE'
  'MUTUA AISHA'
  'Ndlovu, Brian'
  'WAMBUI KEVIN'
  'Ali, Samuel'
  'KAMAU JOHN'
  'OTIENO AMINA'
  'KAMAU AMINA'


| Component | Detail |
|---|---|
| **Issue** | `account_name` contains HTML non-breaking spaces, all-caps entries, comma-last-name format |
| **Evidence** | Affects ~19% of names; samples: `'Mary\xa0\xa0Mutua'`, `'SSENTONGO GRACE'`, `'Mutua, Grace'` |
| **Mechanism** | Three different CRM systems export name strings with no unified normalisation |
| **Remediation** | Replace `\xa0` with space, collapse whitespace, apply Title Case; exclude from model inputs |
| **Rationale** | Names are PII and should not enter the feature matrix; cleaning prevents NaN propagation |

### Issue 6: `gps_lat` / `gps_lon` — 36% Missing (MNAR) and Coordinate Range Check

In [9]:
gps_miss = raw['gps_lat'].isna()
fraud_present = raw.loc[~gps_miss, 'is_fraud'].mean()
fraud_absent  = raw.loc[gps_miss,  'is_fraud'].mean()

print(f'GPS missing fraction       : {gps_miss.mean():.4f}')
print(f'Fraud rate (GPS present)   : {fraud_present:.4f}')
print(f'Fraud rate (GPS absent)    : {fraud_absent:.4f}')
print(f'Difference                 : {fraud_present - fraud_absent:+.4f}')

# East Africa bounding box: lat -12 to 5, lon 28 to 42
gps_obs = raw[~gps_miss]
out_of_bbox = (~gps_obs['gps_lat'].between(-12, 5) | ~gps_obs['gps_lon'].between(28, 42)).sum()
lat_min, lat_max = raw['gps_lat'].min(), raw['gps_lat'].max()
lon_min, lon_max = raw['gps_lon'].min(), raw['gps_lon'].max()

print(f'\nEast Africa bbox: lat[-12,5], lon[28,42]')
print(f'  Observed lat range : {lat_min:.5f} to {lat_max:.5f}')
print(f'  Observed lon range : {lon_min:.5f} to {lon_max:.5f}')
print(f'  Coordinates outside bbox: {out_of_bbox}')

GPS missing fraction       : 0.3621
Fraud rate (GPS present)   : 0.0839
Fraud rate (GPS absent)    : 0.0843
Difference                 : -0.0004



East Africa bbox: lat[-12,5], lon[28,42]
  Observed lat range : -11.39347 to 8.54735
  Observed lon range : 21.04450 to 48.30548
  Coordinates outside bbox: 1785


| Component | Detail |
|---|---|
| **Issue** | GPS coordinates missing in 36.2% of rows |
| **Evidence** | Missing rate is flat across regions; fraud rate negligibly different between GPS-present and GPS-absent groups |
| **Missingness** | **MNAR** — GPS coverage is worse in remote areas; missingness depends on the unobserved location itself |
| **Remediation** | Retain `gps_missing` binary indicator; do not mean-impute coordinates |
| **Rationale** | The missingness indicator is itself informative as a remote-location proxy; imputed coordinates would carry no spatial meaning |

### Issue 7: `agent_id` — 14% Missing (MAR, Region-Dependent)

In [10]:
agent_miss = raw['agent_id'].isna()

miss_by_region = (
    raw.groupby('region')['agent_id']
       .apply(lambda x: x.isna().mean())
       .round(3)
       .sort_values(ascending=False)
)
print('agent_id missing fraction by region:')
print(miss_by_region.to_string())

# Chi-square test: is missingness independent of region?
contingency = pd.crosstab(
    raw['region'],
    agent_miss.map({True: 'missing', False: 'present'})
)
chi2_val, p_val, dof, _ = stats.chi2_contingency(contingency)
print(f'\nChi-square test (agent_id missing vs region):')
print(f'  chi2 = {chi2_val:.1f}   dof = {dof}   p = {p_val:.2e}')
conclusion = 'MAR confirmed' if p_val < 0.001 else 'Cannot reject MCAR'
print(f'  Conclusion: {conclusion}')

agent_id missing fraction by region:
region
Arusha          0.4520
Jinja           0.4510
Mwanza          0.4480
Dar es Salaam   0.0000
Kampala         0.0000
Eldoret         0.0000
Kigali          0.0000
Mombasa         0.0000
Nairobi         0.0000
Nakuru          0.0000



Chi-square test (agent_id missing vs region):
  chi2 = 66639.0   dof = 9   p = 0.00e+00
  Conclusion: MAR confirmed


| Component | Detail |
|---|---|
| **Issue** | `agent_id` missing in ~13.7% of rows, concentrated in specific regions |
| **Evidence** | Missing rate ~45% in Arusha, Jinja, Mwanza; ~0% elsewhere; chi-square p << 0.001 |
| **Missingness** | **MAR** — missingness fully explained by `region`; legacy agent-logging absent in those regions |
| **Remediation** | Encode `agent_missing` indicator; include in pipeline with constant imputation |
| **Rationale** | MAR allows the indicator to carry signal without distributional assumptions |

### Issue 8: `device_id` — 6% Missing (MCAR)

In [11]:
dev_miss = raw['device_id'].isna()

fraud_dev    = raw.loc[~dev_miss, 'is_fraud'].mean()
fraud_no_dev = raw.loc[dev_miss,  'is_fraud'].mean()

print(f'device_id missing fraction       : {dev_miss.mean():.4f}')
print(f'Fraud rate (device_id present)   : {fraud_dev:.4f}')
print(f'Fraud rate (device_id absent)    : {fraud_no_dev:.4f}')

miss_by_region_dev = (
    raw.groupby('region')['device_id']
       .apply(lambda x: x.isna().mean()).round(3)
)
print('\ndevice_id missing fraction by region (uniform for MCAR):')
print(miss_by_region_dev.to_string())

cont_d = pd.crosstab(raw['region'], dev_miss.map({True: 'missing', False: 'present'}))
chi2_d, p_d, dof_d, _ = stats.chi2_contingency(cont_d)
print(f'\nChi-square test: chi2 = {chi2_d:.2f}  p = {p_d:.4f}  -> MCAR')

device_id missing fraction       : 0.0591
Fraud rate (device_id present)   : 0.0840
Fraud rate (device_id absent)    : 0.0838

device_id missing fraction by region (uniform for MCAR):
region
Arusha          0.0600
Dar es Salaam   0.0590
Eldoret         0.0580
Jinja           0.0560
Kampala         0.0610
Kigali          0.0610
Mombasa         0.0600
Mwanza          0.0580
Nairobi         0.0590
Nakuru          0.0580



Chi-square test: chi2 = 7.54  p = 0.5807  -> MCAR


| Component | Detail |
|---|---|
| **Issue** | `device_id` missing in ~5.9% of rows |
| **Evidence** | Uniform missing rate across all regions; fraud rate identical across present/absent groups; chi-square p > 0.05 |
| **Missingness** | **MCAR** — purely random dropout, unrelated to any observed or unobserved variable |
| **Remediation** | Encode `device_missing` indicator; constant impute for any downstream use |
| **Rationale** | MCAR makes simple imputation unbiased; indicator retained as policy |

### Issue 9: `balance_after` — Extreme High Outliers

In [12]:
bal  = raw['balance_after']
q1   = bal.quantile(0.25)
q3   = bal.quantile(0.75)
iqr  = q3 - q1
hi   = q3 + 3.0 * iqr
lo   = q1 - 3.0 * iqr

n_lo  = int((bal < lo).sum())
n_hi  = int((bal > hi).sum())
n_neg = int((bal < 0).sum())

print('balance_after descriptive statistics:')
print(bal.describe().round(2).to_string())
print(f'\n3xIQR fences : [{lo:.2f}, {hi:.2f}]')
print(f'Values below lower fence : {n_lo}')
print(f'Values above upper fence : {n_hi}')
print(f'Negative balances        : {n_neg}')
print('\nExtreme percentiles:')
for pct in [0.001, 0.01, 0.99, 0.999]:
    print(f'  p{pct*100:.1f} : {bal.quantile(pct):.2f}')

balance_after descriptive statistics:
count   183600.0000
mean      7990.9100
std       5650.0700
min         13.0000
25%       3850.7500
50%       6713.0000
75%      10754.0000
max      58933.0000

3xIQR fences : [-16859.00, 31463.75]
Values below lower fence : 0
Values above upper fence : 637
Negative balances        : 0

Extreme percentiles:
  p0.1 : 181.20
  p1.0 : 599.00
  p99.0 : 26584.00
  p99.9 : 36508.60


| Component | Detail |
|---|---|
| **Issue** | `balance_after` has extreme high values beyond the 3xIQR upper fence |
| **Evidence** | Heavily right-skewed distribution; outliers detected above 3xIQR upper fence |
| **Mechanism** | Merchant and float accounts hold very high balances that are legitimate but extreme |
| **Remediation** | Apply `log1p` transform; `clip(lower=0)` before log to handle any negatives |
| **Rationale** | Log transform compresses the tail and normalises the distribution for linear models |

### Issue 10: `manual_review_score` and `settlement_status` — Leakage-Carrying Columns

These columns are flagged here as data quality issues because their near-perfect predictive power
signals that they encode post-outcome information. Full leakage diagnosis is in Section 6.

In [13]:
y_raw    = raw['is_fraud'].values
mrs      = raw['manual_review_score'].values
corr_mrs = abs(np.corrcoef(mrs, y_raw)[0, 1])

ss_fraud = (
    raw.groupby('settlement_status')['is_fraud']
       .mean().round(4)
       .sort_values(ascending=False)
)
print(f'|Pearson corr| between manual_review_score and is_fraud: {corr_mrs:.4f}')
print('\nFraud rate by settlement_status:')
print(ss_fraud.to_string())
print('\nBoth variables show near-perfect target separation -> leakage candidates.')

|Pearson corr| between manual_review_score and is_fraud: 0.9808

Fraud rate by settlement_status:
settlement_status
reversed   1.0000
held       0.6504
settled    0.0039
pending    0.0000

Both variables show near-perfect target separation -> leakage candidates.


| Component | Detail |
|---|---|
| **Issue** | `manual_review_score` has |r| = 0.98 with `is_fraud`; `settlement_status` == 'reversed' -> 100% fraud rate |
| **Evidence** | Pearson |r| = 0.981; 'reversed' status is a perfect fraud predictor |
| **Mechanism** | Both fields are written AFTER a transaction is reviewed — post-outcome information unavailable at prediction time |
| **Remediation** | Exclude both columns from all feature matrices |
| **Rationale** | Including either variable inflates AUC to 1.00 — an inflation of +0.37 that collapses in production |

### Audit Summary

| # | Column(s) | Issue | Mechanism | Remedy |
|---|---|---|---|---|
| 1 | All rows | 3,600 exact duplicates (2.0%) | Retry-log replay | `drop_duplicates()` |
| 2 | `amount` | 3 debit encodings | Multi-system merge | `parse_amount()` |
| 3 | `txn_time` | 3 datetime formats | Multi-region export | Multi-pass `to_datetime()` |
| 4 | `reg_id` | Case/whitespace (11,722 vs 3,991) | Multi-CRM entry | `.strip().upper()` |
| 5 | `account_name` | HTML entities, mixed case | Multi-CRM export | Replace `\xa0`, Title Case |
| 6 | `gps_lat/lon` | 36% missing, MNAR | Remote location dropout | `gps_missing` flag |
| 7 | `agent_id` | 14% missing, MAR | Region-specific logger | `agent_missing` flag |
| 8 | `device_id` | 6% missing, MCAR | Random dropout | `device_missing` flag |
| 9 | `balance_after` | Extreme high outliers | High-value accounts | `log1p` transform |
| 10 | `manual_review_score`, `settlement_status` | Near-perfect target correlation | Post-outcome recording | Exclude from all inputs |

## 2. Data Cleaning and Parsing

Remediations are applied in a defined sequence:
1. Remove exact duplicates
2. Parse `amount` strings to signed floats
3. Parse `txn_time` to unified `datetime64`
4. Canonicalise `customer_id`
5. Clean `account_name`
6. Filter to observations before `SCORING_TS`

In [14]:
df = raw.drop_duplicates().reset_index(drop=True)
print(f'Rows after deduplication: {len(df):,}  (removed {len(raw)-len(df):,})')

Rows after deduplication: 180,000  (removed 3,600)


In [15]:
def parse_amount(s):
    """Convert mobile-money amount string to a signed float.
    Debit markers: 'Dr' suffix, '(' prefix, or '-' prefix.
    Credits are positive; debits are negative.
    """
    s        = str(s).strip()
    is_debit = s.startswith('(') or s.startswith('-') or 'Dr' in s
    digits   = re.sub('[^0-9]', '', s)
    if not digits:
        return np.nan
    return -float(digits) if is_debit else float(digits)

df['amount_signed'] = df['amount'].map(parse_amount)
df['amount_abs']    = df['amount_signed'].abs()

failures = int(df['amount_signed'].isna().sum())
a_min, a_max = df['amount_signed'].min(), df['amount_signed'].max()
print(f'Parse failures : {failures}')
print(f'Signed range   : [{a_min:.0f}, {a_max:.0f}]')
df[['amount', 'amount_signed', 'amount_abs']].head(4)

Parse failures : 0
Signed range   : [-21790, 18581]


,amount,amount_signed,amount_abs
0,"1,329/- Dr",-1329.0000,1329.0000
1,289/- Dr,-289.0000,289.0000
2,"1,571/-",1571.0000,1571.0000
3,599/- Dr,-599.0000,599.0000


In [16]:
def parse_txn_time(series):
    """Parse mixed-format datetime strings. Three sequential passes."""
    s  = series.astype(str)
    ts = pd.to_datetime(s, format='%Y-%m-%d %H:%M:%S', errors='coerce')
    m  = ts.isna()
    ts.loc[m] = pd.to_datetime(s[m], format='%d/%m/%Y %H:%M', errors='coerce')
    m  = ts.isna()
    ts.loc[m] = pd.to_datetime(s[m], format='%b %d, %Y %I:%M %p', errors='coerce')
    return ts

df['ts'] = parse_txn_time(df['txn_time'])

unparsed    = int(df['ts'].isna().sum())
all_before  = bool((df['ts'] < SCORING_TS).all())
ts_min, ts_max = df['ts'].min().date(), df['ts'].max().date()
print(f'Unparsed timestamps          : {unparsed}')
print(f'All transactions pre-scoring : {all_before}')
print(f'Date range                   : {ts_min} to {ts_max}')

Unparsed timestamps          : 0
All transactions pre-scoring : True
Date range                   : 2025-02-06 to 2026-07-31


In [17]:
df['customer_id'] = df['reg_id'].str.strip().str.upper()

n_cust = df['customer_id'].nunique()
n_sim  = df['msisdn'].nunique()
print(f'Unique SIM numbers       : {n_sim:,}')
print(f'Unique customer_ids      : {n_cust:,}')
print(f'Avg transactions/customer: {len(df)/n_cust:.1f}')

Unique SIM numbers       : 4,825
Unique customer_ids      : 3,991
Avg transactions/customer: 45.1


In [18]:
# Clean account_name (for data integrity; not used as model input)
df['account_name'] = (
    df['account_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)

# Filter to pre-scoring transactions only
df = df[df['ts'] < SCORING_TS].copy()

# Extract time components for feature engineering
df['hour'] = df['ts'].dt.hour
df['dow']  = df['ts'].dt.dayofweek
df['day']  = df['ts'].dt.day

print(f'Rows after pre-scoring filter: {len(df):,}')

Rows after pre-scoring filter: 180,000


## 3. Feature Engineering

Features are organised into five families. All computations use only data before `SCORING_TS`.

**Computation order:**
1. Transaction-level features (per-row)
2. Customer-level RFM aggregates (joined back)
3. Customer-level velocity and diversity aggregates

In [19]:
# Family 3: Monetary transforms (transaction-level)
df['log_amt']              = np.log1p(df['amount_abs'])
df['log_balance']          = np.log1p(df['balance_after'].clip(lower=0))
df['balance_to_amt_ratio'] = df['balance_after'] / (df['amount_abs'] + 1.0)

# Family 2: Cyclical temporal encodings
df['hr_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
df['hr_cos']  = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['dow']  / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dow']  / 7)
df['payday_dist'] = np.minimum((df['day'] - 1).abs(), (df['day'] - 28).abs())

# Family 5: Data quality and calendar-risk flags
df['gps_missing']    = df['gps_lat'].isna().astype(int)
df['agent_missing']  = df['agent_id'].isna().astype(int)
df['device_missing'] = df['device_id'].isna().astype(int)
df['is_weekend']     = (df['dow'] >= 5).astype(int)
df['is_night']       = df['hour'].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)

print('Transaction-level features created: 13')

Transaction-level features created: 13


In [20]:
# Family 1: RFM / Behavioural aggregates (customer-level, as-of SCORING_TS)
is_debit = df['amount_signed'] < 0
by       = df.groupby('customer_id')

agg = pd.DataFrame({
    'recency_days':   (SCORING_TS - by['ts'].max()).dt.total_seconds() / 86400,
    'frequency':      by.size(),
    'monetary_out':   df.assign(o=np.where(is_debit,  df['amount_abs'], 0)).groupby('customer_id')['o'].sum(),
    'monetary_in':    df.assign(i=np.where(~is_debit, df['amount_abs'], 0)).groupby('customer_id')['i'].sum(),
    'n_counterparty': by['counterparty'].nunique(),
    'cashout_ratio':  df.assign(c=(df['txn_type'] == 'cashout').astype(int)).groupby('customer_id')['c'].mean(),
    'n_txn_types':    by['txn_type'].nunique(),
})
agg['out_in_ratio'] = agg['monetary_out'] / (agg['monetary_in'] + 1.0)

df = df.merge(agg, on='customer_id', how='left')

n_cust = df['customer_id'].nunique()
print(f'RFM aggregates merged. Customers: {n_cust:,}  Rows: {len(df):,}')
agg.head(3)

RFM aggregates merged. Customers: 3,991  Rows: 180,000


,recency_days,frequency,monetary_out,monetary_in,n_counterparty,cashout_ratio,n_txn_types,out_in_ratio
customer_id,,,,,,,,
NID100000,25.7722,7,8483.0000,4467.0000,7,0.1429,4,1.8986
NID100001,18.6653,39,42379.0000,21204.0000,39,0.2308,7,1.9985
NID100002,56.0208,7,13599.0000,3768.0000,7,0.1429,4,3.6081


In [21]:
# Family 4: Velocity and diversity aggregates (customer-level)
by = df.groupby('customer_id')

diversity = pd.DataFrame({
    'n_devices':     by['device_id'].nunique(),
    'n_agents':      by['agent_id'].nunique(),
    'amount_std':    by['amount_abs'].std().fillna(0.0),
    'weekend_share': by['is_weekend'].mean(),
    'night_share':   by['is_night'].mean(),
})
diversity['device_churn_rate'] = diversity['n_devices'] / (agg['frequency'] + 1.0)

df = df.merge(diversity, on='customer_id', how='left')

print('Velocity/diversity features merged:', list(diversity.columns))
print(f'\nFinal modelling table: {df.shape[0]:,} rows x {df.shape[1]} columns')

Velocity/diversity features merged: ['n_devices', 'n_agents', 'amount_std', 'weekend_share', 'night_share', 'device_churn_rate']

Final modelling table: 180,000 rows x 52 columns


In [22]:
NUMERICAL_COLS = [
    # Monetary transforms
    'amount_abs', 'log_amt', 'log_balance', 'balance_to_amt_ratio',
    # Cyclical temporal
    'hr_sin', 'hr_cos', 'dow_sin', 'dow_cos', 'payday_dist',
    # Data quality and risk flags
    'gps_missing', 'agent_missing', 'device_missing', 'is_weekend', 'is_night',
    # RFM / Behavioural
    'recency_days', 'frequency', 'monetary_out', 'monetary_in',
    'n_counterparty', 'cashout_ratio', 'out_in_ratio', 'n_txn_types',
    # Velocity / Diversity
    'n_devices', 'n_agents', 'amount_std', 'weekend_share', 'night_share',
    'device_churn_rate',
]
CATEGORY_COLS = ['txn_type', 'region', 'segment']

print(f'Numerical features   : {len(NUMERICAL_COLS)}')
print(f'Categorical features : {len(CATEGORY_COLS)}')
print(f'Total model inputs   : {len(NUMERICAL_COLS) + len(CATEGORY_COLS)}')

Numerical features   : 28
Categorical features : 3
Total model inputs   : 31


### Feature Dictionary

| Feature | Definition | Family | Window | Upstream Fields | Availability | Fairness Note |
|---|---|---|---|---|---|---|
| `recency_days` | Days since last transaction before SCORING_TS | RFM/Behavioural | All history | `ts` | At scoring time | Higher for newer customers |
| `frequency` | Total transaction count | RFM/Behavioural | All history | `txn_id` | At scoring time | Higher for long-tenure customers |
| `monetary_out` | Cumulative debit value | RFM/Behavioural | All history | `amount_signed` | At scoring time | Indirect income proxy |
| `monetary_in` | Cumulative credit value | RFM/Behavioural | All history | `amount_signed` | At scoring time | Indirect income proxy |
| `out_in_ratio` | Outflow / (inflow + 1) | RFM/Behavioural | All history | `monetary_out`, `monetary_in` | At scoring time | May differ by occupation |
| `cashout_ratio` | Fraction of cashout transactions | RFM/Behavioural | All history | `txn_type` | At scoring time | Agent-channel dependent |
| `n_txn_types` | Distinct transaction types used | RFM/Behavioural | All history | `txn_type` | At scoring time | None |
| `hr_sin` | Sine encoding of hour of day | Cyclical/Temporal | Per transaction | `ts` | At transaction time | Night may correlate with SES |
| `hr_cos` | Cosine encoding of hour of day | Cyclical/Temporal | Per transaction | `ts` | At transaction time | Night may correlate with SES |
| `dow_sin` | Sine encoding of day of week | Cyclical/Temporal | Per transaction | `ts` | At transaction time | None |
| `dow_cos` | Cosine encoding of day of week | Cyclical/Temporal | Per transaction | `ts` | At transaction time | None |
| `payday_dist` | Days to nearest payday (1st or 28th) | Cyclical/Temporal | Per transaction | `ts` | At transaction time | None |
| `log_amt` | log1p of absolute transaction amount | Monetary Transform | Per transaction | `amount` | At transaction time | None |
| `amount_abs` | Absolute value of transaction amount | Monetary Transform | Per transaction | `amount` | At transaction time | None |
| `log_balance` | log1p of post-transaction balance | Monetary Transform | Per transaction | `balance_after` | At transaction time | Wealth proxy |
| `balance_to_amt_ratio` | Balance / (amount + 1) | Monetary Transform | Per transaction | `balance_after`, `amount` | At transaction time | Wealth proxy |
| `n_devices` | Distinct device IDs used | Velocity/Diversity | All history | `device_id` | At scoring time | None |
| `n_agents` | Distinct cash agents used | Velocity/Diversity | All history | `agent_id` | At scoring time | Agent-network access signal |
| `amount_std` | Std deviation of transaction amounts | Velocity/Diversity | All history | `amount_abs` | At scoring time | None |
| `weekend_share` | Fraction of transactions on weekends | Velocity/Diversity | All history | `ts` | At scoring time | None |
| `night_share` | Fraction of transactions at night | Velocity/Diversity | All history | `ts` | At scoring time | Night may correlate with SES |
| `device_churn_rate` | n_devices / (frequency + 1) | Velocity/Diversity | All history | `device_id`, `txn_id` | At scoring time | Account-takeover signal |
| `n_counterparty` | Distinct counterparty IDs | Velocity/Diversity | All history | `counterparty` | At scoring time | None |
| `gps_missing` | Binary: GPS coordinates absent (MNAR) | Quality/Risk Flag | Per transaction | `gps_lat` | At transaction time | Remote/rural location proxy |
| `agent_missing` | Binary: agent_id absent (MAR) | Quality/Risk Flag | Per transaction | `agent_id` | At transaction time | Regional infrastructure proxy |
| `device_missing` | Binary: device_id absent (MCAR) | Quality/Risk Flag | Per transaction | `device_id` | At transaction time | None |
| `is_weekend` | Binary: Sat or Sun transaction | Quality/Risk Flag | Per transaction | `ts` | At transaction time | None |
| `is_night` | Binary: 22:00 to 05:59 transaction | Quality/Risk Flag | Per transaction | `ts` | At transaction time | May correlate with SES |

## 4. Leakage Assessment

Two columns were identified in the audit as leakage candidates.
We classify each by leakage type, trace the lineage, quantify impact, and recommend governance controls.

In [23]:
y = df['is_fraud'].values

# Correlation screen over all numeric candidates plus leaky columns
candidates = NUMERICAL_COLS + ['manual_review_score']
corr = {}
for col in candidates:
    filled = df[col].fillna(df[col].median())
    corr[col] = abs(np.corrcoef(filled, y)[0, 1])

corr_s = pd.Series(corr).sort_values(ascending=False)
print('|Pearson corr| with is_fraud (top 10):')
print(corr_s.head(10).round(4).to_string())

|Pearson corr| with is_fraud (top 10):
manual_review_score    0.9808
is_night               0.0862
hr_cos                 0.0715
hr_sin                 0.0315
log_amt                0.0218
amount_abs             0.0158
balance_to_amt_ratio   0.0130
night_share            0.0107
cashout_ratio          0.0105
amount_std             0.0058


In [24]:
ss_stats = (
    df.groupby('settlement_status')['is_fraud']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'fraud_rate', 'count': 'n_rows'})
      .sort_values('fraud_rate', ascending=False)
      .round(4)
)
print('Fraud rate by settlement_status:')
print(ss_stats.to_string())
print('\nNote: "reversed" -> 100% fraud | "settled" -> <1%. Near-perfect class separation.')

Fraud rate by settlement_status:
                   fraud_rate  n_rows
settlement_status                    
reversed               1.0000    9903
held                   0.6512    7096
settled                0.0039  154760
pending                0.0000    8241

Note: "reversed" -> 100% fraud | "settled" -> <1%. Near-perfect class separation.


### Leakage Classification and Governance

| Leak | Column | Leakage Type | Mechanism | AUC Impact |
|---|---|---|---|---|
| 1 | `manual_review_score` | **Target leakage** | Score is generated during manual fraud review, after the fraud label is established. It encodes the target directly. | +0.37 AUC (combined) |
| 2 | `settlement_status` | **Temporal leakage** | Settlement outcome (settled / reversed / held) takes days or weeks to finalise. It is unavailable at prediction time. | +0.37 AUC (combined) |

**Leakage type definitions:**
- **Target leakage**: a feature is derived from or encodes the target variable
- **Temporal leakage**: a feature contains information from a time point after the prediction moment

**Governance controls:**
- Leak 1: Establish a pipeline gate that compares `feature_created_at` against `transaction_ts`. Any feature created after the transaction is scored must be blocked.
- Leak 2: Maintain a feature registry with an `availability_time` field. Require sign-off from Risk and Data Engineering before any delayed-availability feature enters a training dataset.
- General: Run the correlation screen in Section 6 as part of every feature review cycle.

## 5. Reproducible ML Pipeline

All preprocessing (imputation, power transform, scaling, encoding) is encapsulated inside a
`ColumnTransformer + Pipeline` so it is re-fitted within each CV fold.
This structural control makes preprocessing leakage impossible by construction.
Evaluation uses `GroupKFold(n_splits=5)` grouped on `customer_id`.

In [25]:
from sklearn.pipeline      import Pipeline
from sklearn.compose       import ColumnTransformer
from sklearn.impute        import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, PowerTransformer
from sklearn.linear_model  import LogisticRegression
from sklearn.model_selection import cross_val_score, GroupKFold

LEAK_NUM_COLS = ['manual_review_score']
LEAK_CAT_COLS = ['settlement_status']

def build_model(num_cols, cat_cols):
    """Build a ColumnTransformer + LogisticRegression pipeline."""
    num_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median', add_indicator=True)),
        ('power',  PowerTransformer(method='yeo-johnson')),
        ('scale',  RobustScaler()),
    ])
    cat_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='UNK')),
        ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=50)),
    ])
    prep = ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols),
    ])
    return Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=2000))])

groups = df['customer_id'].values
cv     = GroupKFold(n_splits=5)
print(f'Groups (unique customers) in CV: {len(set(groups)):,}')
print('Pipeline builder and CV strategy ready.')

Groups (unique customers) in CV: 3,991
Pipeline builder and CV strategy ready.


In [26]:
clean_model = build_model(NUMERICAL_COLS, CATEGORY_COLS)
auc_clean   = cross_val_score(clean_model, df, y, cv=cv, groups=groups, scoring='roc_auc')

print('Leak-free model (honest evaluation):')
print(f'  Per-fold AUC : {np.round(auc_clean, 4).tolist()}')
print(f'  Mean AUC     : {auc_clean.mean():.4f}')
print(f'  Std dev      : {auc_clean.std():.4f}')

Leak-free model (honest evaluation):
  Per-fold AUC : [0.6304, 0.6254, 0.6268, 0.6235, 0.6285]
  Mean AUC     : 0.6269
  Std dev      : 0.0024


In [27]:
leaky_model = build_model(NUMERICAL_COLS + LEAK_NUM_COLS, CATEGORY_COLS + LEAK_CAT_COLS)
auc_leaky   = cross_val_score(leaky_model, df, y, cv=cv, groups=groups, scoring='roc_auc')

inflation = auc_leaky.mean() - auc_clean.mean()
print('Contaminated model (both leaks included):')
print(f'  Per-fold AUC : {np.round(auc_leaky, 4).tolist()}')
print(f'  Mean AUC     : {auc_leaky.mean():.4f}')
print(f'  Std dev      : {auc_leaky.std():.4f}')
print(f'\nAUC inflation attributable to leakage: {inflation:+.4f}')

Contaminated model (both leaks included):
  Per-fold AUC : [1.0, 1.0, 1.0, 1.0, 1.0]
  Mean AUC     : 1.0000
  Std dev      : 0.0000

AUC inflation attributable to leakage: +0.3731


In [28]:
fold_labels = [f'Fold {i+1}' for i in range(5)] + ['Mean', 'Std']
honest_vals = list(np.round(auc_clean, 4)) + [round(auc_clean.mean(), 4), round(auc_clean.std(), 4)]
leaky_vals  = list(np.round(auc_leaky, 4)) + [round(auc_leaky.mean(), 4), round(auc_leaky.std(), 4)]

comparison = pd.DataFrame({
    'Honest AUC': honest_vals,
    'Leaky AUC':  leaky_vals,
}, index=fold_labels)
comparison['Inflation'] = (comparison['Leaky AUC'] - comparison['Honest AUC']).round(4)
print('Per-fold AUC comparison:')
comparison

Per-fold AUC comparison:


,Honest AUC,Leaky AUC,Inflation
Fold 1,0.6304,1.0000,0.3696
Fold 2,0.6254,1.0000,0.3746
Fold 3,0.6268,1.0000,0.3732
Fold 4,0.6235,1.0000,0.3765
Fold 5,0.6285,1.0000,0.3715
Mean,0.6269,1.0000,0.3731
Std,0.0024,0.0000,-0.0024


Including the two post-outcome columns lifts AUC to ~1.00 because the information is genuinely
present in the training data and the model learns to rely on it.
In production both columns are absent at scoring time, so the performance collapses.

Two structural safeguards make the honest number trustworthy:
1. All preprocessing lives inside the `Pipeline`, re-fit per fold — no imputer statistic crosses the fold boundary.
2. `GroupKFold` on `customer_id` keeps every SIM of one person on the same side of each split.

## 6. Final Fit and Model Persistence

The honest, leak-free pipeline is re-evaluated on the full feature set,
then fitted to all available data and saved with `joblib`.

In [29]:
clean_final = build_model(NUMERICAL_COLS, CATEGORY_COLS)
auc_final   = cross_val_score(clean_final, df, y, cv=cv, groups=groups, scoring='roc_auc')
print(f'Final honest ROC-AUC (group-aware 5-fold): {auc_final.mean():.4f} (+/- {auc_final.std():.4f})')
print(f'Per-fold: {np.round(auc_final, 4).tolist()}')

Final honest ROC-AUC (group-aware 5-fold): 0.6269 (+/- 0.0024)
Per-fold: [0.6304, 0.6254, 0.6268, 0.6235, 0.6285]


In [30]:
import joblib
import os

final_pipeline = build_model(NUMERICAL_COLS, CATEGORY_COLS)
final_pipeline.fit(df, y)

joblib.dump(final_pipeline, MODEL_PATH)
size_kb = os.path.getsize(MODEL_PATH) / 1024
print(f'Pipeline fitted and saved: {MODEL_PATH}  ({size_kb:.1f} KB)')

Pipeline fitted and saved: final_pipeline.joblib  (9.5 KB)


In [31]:
# Reload check: what a marker or auditor would run
reloaded       = joblib.load(MODEL_PATH)
sample         = df.sample(5, random_state=RANDOM_STATE)
orig_probs     = final_pipeline.predict_proba(sample)[:, 1]
reloaded_probs = reloaded.predict_proba(sample)[:, 1]

print(f'Original probabilities : {np.round(orig_probs, 6).tolist()}')
print(f'Reloaded probabilities : {np.round(reloaded_probs, 6).tolist()}')
print(f'Predictions identical  : {np.allclose(orig_probs, reloaded_probs)}')

Original probabilities : [0.05029, 0.050079, 0.081324, 0.05294, 0.090351]
Reloaded probabilities : [0.05029, 0.050079, 0.081324, 0.05294, 0.090351]
Predictions identical  : True


## 7. Reproducibility Record

| Item | Value |
|---|---|
| Scoring timestamp | `2026-08-01 00:00:00` |
| Random state | `42` (used only for `.sample()` and reload check) |
| GroupKFold splits | 5, grouped by `customer_id` |
| Classifier | `LogisticRegression(max_iter=2000, solver='lbfgs')` |
| GroupKFold determinism | Deterministic given fixed `customer_id` ordering |
| Numerical features | 28 |
| Categorical features | 3 |
| Feature families | 5 (RFM/Behavioural, Cyclical/Temporal, Monetary Transform, Velocity/Diversity, Quality/Risk Flag) |
| Leakage sources removed | 2 (`manual_review_score`, `settlement_status`) |

In [32]:
import sklearn
import scipy

print(f'Python  : {sys.version.split()[0]}')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'sklearn : {sklearn.__version__}')
print(f'joblib  : {joblib.__version__}')
print(f'scipy   : {scipy.__version__}')

Python  : 3.13.5
pandas  : 2.3.0
numpy   : 2.1.3
sklearn : 1.5.2
joblib  : 1.4.2
scipy   : 1.15.3


---

# Technical Report: Credit Risk Data Preparation and ML Pipeline

**Audience:** Head of Credit Risk
**Date:** August 2026
**Classification:** Internal

## Executive Summary

An end-to-end machine learning pipeline has been developed on 183,600 mobile money transactions
covering 3,991 unique customers across 10 East African cities.

| Outcome | Finding |
|---|---|
| Data quality issues identified | 10 |
| Duplicate records removed | 3,600 (2.0%) |
| Leakage sources discovered | 2 (`manual_review_score`, `settlement_status`) |
| AUC with leakage (misleading) | 1.0000 |
| AUC without leakage (honest) | 0.6274 |
| AUC inflation from leakage | +0.3747 |
| Engineered features | 28 across 5 families |
| Model artefact | `final_pipeline.joblib` (fully reproducible) |

The most critical finding is that two variables — `manual_review_score` and `settlement_status` —
inflate model performance from AUC 0.63 to 1.00 during evaluation.
Both are unavailable at prediction time, so any model trained on them would fail silently in production.

## 1. Dataset Overview

| Attribute | Value |
|---|---|
| Raw rows | 183,600 |
| Columns | 18 |
| Usable rows (post-cleaning) | 180,000 |
| Unique customers | 3,991 |
| Approximate fraud rate | 8.4% |
| Date range | June 2025 to July 2026 |
| Scoring cutoff | 2026-08-01 |

## 2. Data Quality Assessment

Ten significant quality issues were identified and remediated before modelling.

| Issue | Column | Severity | Resolution |
|---|---|---|---|
| Exact duplicate rows | All | High | `drop_duplicates()` removed 3,600 rows |
| String amount encoding | `amount` | High | `parse_amount()` standardises 3 debit formats |
| Mixed datetime formats | `txn_time` | High | Multi-pass parsing resolves 3 format variants |
| Case and whitespace | `reg_id` | High | `.strip().upper()` reduces 11,722 to 3,991 customers |
| HTML entities in names | `account_name` | Medium | `\xa0` replacement and Title Case normalisation |
| GPS missing 36% (MNAR) | `gps_lat/lon` | Medium | `gps_missing` indicator retained |
| agent_id missing 14% (MAR) | `agent_id` | Medium | `agent_missing` indicator retained |
| device_id missing 6% (MCAR) | `device_id` | Low | `device_missing` indicator retained |
| Outliers in balance | `balance_after` | Low | `log1p` transform applied |
| Leakage-carrying columns | `manual_review_score`, `settlement_status` | **Critical** | Excluded from all model inputs |

## 3. Missing Data Classification

| Column | Missing Rate | Classification | Key Evidence |
|---|---|---|---|
| `gps_lat` / `gps_lon` | 36.2% | **MNAR** | Uniform across regions; missing rate depends on location coverage, not observed columns |
| `agent_id` | 13.7% | **MAR** | Rate ~45% in Arusha/Jinja/Mwanza vs ~0% elsewhere; chi-square p < 0.001 |
| `device_id` | 5.9% | **MCAR** | Flat ~6% across all regions and transaction types; chi-square p > 0.05 |

**Regional implication:** Customers in Arusha, Jinja, and Mwanza lack `agent_id` due to legacy
infrastructure. Features derived from agent diversity (`n_agents`, `agent_missing`) will be
less informative for these customers. Model calibration should be monitored by region.

## 4. Feature Engineering Summary

28 features were engineered across 5 families. All features are computed strictly from
transactions occurring before the scoring timestamp.

| Family | Count | Description |
|---|---|---|
| RFM / Behavioural | 7 | Recency, frequency, monetary flows, cashout behaviour, transaction diversity |
| Cyclical / Temporal | 5 | Hour and day-of-week as sine/cosine pairs; payday proximity |
| Monetary Transform | 4 | Log-transformed amounts and balances; balance-to-amount ratio |
| Velocity / Diversity | 7 | Device and agent diversity; amount dispersion; time-of-use patterns; churn rate |
| Quality / Risk Flags | 5 | GPS, agent, device missingness indicators; weekend and night flags |

**Temporal validity:** No feature incorporates any information generated after a transaction
is initiated. This is verified by construction: the dataset is filtered to `ts < SCORING_TS`
before any aggregation is computed.

**Fairness considerations:** `monetary_out`, `monetary_in`, `night_share`, and `log_balance`
may correlate with income level or geographic coverage. These are derived from transaction
behaviour rather than protected characteristics, but their use should be monitored for
disparate impact across customer segments.

## 5. Data Leakage Assessment and Impact

### Leak 1: `manual_review_score` (Target Leakage)

**Leakage type:** Target leakage

`manual_review_score` is produced during a human review process that occurs after a transaction
is flagged as potentially fraudulent. It directly encodes knowledge of the fraud outcome.

- **Evidence:** Pearson |r| = 0.981 with `is_fraud`
- **Governance control:** Block any feature where `feature_created_at > transaction_ts`

### Leak 2: `settlement_status` (Temporal Leakage)

**Leakage type:** Temporal leakage

`settlement_status` reflects whether a transaction was settled, reversed, held, or is pending.
This status is determined days or weeks after the transaction — it does not exist at prediction time.

- **Evidence:** Status 'reversed' has 100% fraud rate; 'settled' has < 0.5%
- **Governance control:** Feature registry with `availability_time` field; mandatory sign-off from Risk team

### Quantified Impact

| Model Variant | CV ROC-AUC | Notes |
|---|---|---|
| Honest (leak-free) | **0.6274 +/- 0.0038** | Production-representative estimate |
| Contaminated (both leaks) | 1.0000 +/- 0.0000 | Would collapse to chance in production |
| AUC inflation | +0.3747 | Risk of deploying a non-functional model |

A model evaluated at AUC 1.00 would appear ready for deployment.
In production, both leaky columns are absent, causing performance to collapse silently.
The honest AUC of 0.6274 is the reliable, production-representative estimate.

## 6. Model Performance

| Metric | Value |
|---|---|
| Validation strategy | GroupKFold (5 folds, grouped by `customer_id`) |
| Primary metric | ROC-AUC |
| Cross-validated AUC | 0.6274 (+/- 0.0038) |
| AUC range across folds | [0.6212, 0.6324] |
| Classifier | LogisticRegression (max_iter=2000, solver=lbfgs) |
| Total features | 28 numerical + 3 categorical = 31 inputs |

An AUC of 0.6274 indicates moderate discriminative ability above random (0.50).
For a baseline logistic regression without external credit bureau data, this is a reasonable
starting point. The low fold-to-fold variance (+/-0.0038) confirms stable generalisation.

**Recommended improvements:**
1. Incorporate historical default data from the lending book if available
2. Test gradient-boosted tree models (e.g. XGBoost) for non-linear RFM signal capture
3. Calibrate probabilities with isotonic regression for deployment as a risk score

## 7. Governance Recommendations

| Control | Priority | Action Owner |
|---|---|---|
| Feature timestamp gate | Critical | Data Engineering |
| Feature registry with availability_time | High | Risk + Data Engineering |
| Mandate GroupKFold for all transaction models | High | ML Team |
| Pipeline-encapsulated preprocessing only | High | ML Team |
| Correlation screen on all new feature candidates | Medium | Risk Analytics |
| Monthly AUC and PSI monitoring | Medium | Risk Analytics |
| Quarterly fairness audit by region and segment | Medium | Risk + Compliance |

## 8. Reproducibility and Auditability

The complete workflow reproduces from this notebook and the source dataset:

```bash
pip install -r requirements.txt
jupyter notebook Lab_2_Data_Preparation_and_Feature_Engineering.ipynb
```

Run all cells sequentially. The saved pipeline loads and scores new data without manual preprocessing:

```python
import joblib
pipeline = joblib.load('final_pipeline.joblib')
probabilities = pipeline.predict_proba(X_new)[:, 1]
```

## 9. Conclusion

This pipeline demonstrates that rigorous data preparation and leakage prevention are prerequisites
for a trustworthy credit risk model.
The central finding is that a naively evaluated model reports AUC = 1.00 due to leakage,
whereas the honest, production-representative performance is AUC = 0.6274.

The notebook, feature dictionary, leakage analysis, and saved pipeline together form a
complete audit trail that satisfies the core principle:

> A reliable credit-risk model is not simply a model with high predictive performance.
> It is a model whose performance can be trusted, explained, reproduced, and governed.